In [ ]:
# ============================================================
# SETTINGS: change these 3 values when you run it for real
# ============================================================
PROJECT_ID  = "your_project_id"      # your GCP project id
BUCKET      = "your_bucket_name"     # GCS bucket with the 3 CSV files (also used as temp bucket)
BQ_DATASET  = "your_bq_dataset"      # BigQuery dataset where the result tables go

# File paths in GCS (built from the bucket name above)
FLIGHTS_PATH  = f"gs://{BUCKET}/flights_sample.csv"
AIRPORTS_PATH = f"gs://{BUCKET}/airports.csv"
AIRLINES_PATH = f"gs://{BUCKET}/airlines.csv"

# Output tables in BigQuery
TABLE_AIRPORT = f"{PROJECT_ID}.{BQ_DATASET}.delay_causes_by_airport"
TABLE_AIRLINE = f"{PROJECT_ID}.{BQ_DATASET}.delay_causes_by_airline"
TABLE_SUMMARY = f"{PROJECT_ID}.{BQ_DATASET}.airport_main_cause"

In [ ]:
# ============================================================
# Create the Spark session 
# ============================================================
from pyspark.sql import SparkSession
from pyspark import SparkConf

sparkConf = SparkConf()
sparkConf.setMaster("spark://spark-master:7077")   # the Spark master container on your VM
sparkConf.setAppName("B2_DelayCauseAnalysis")
sparkConf.set("spark.driver.memory", "2g")
sparkConf.set("spark.executor.cores", "1")
sparkConf.set("spark.driver.cores", "1")

spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()

# Let Spark read gs:// paths
conf = spark.sparkContext._jsc.hadoopConfiguration()
conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")

# Temporary bucket the BigQuery connector uses when writing
spark.conf.set("temporaryGcsBucket", BUCKET)

1.Read the flights file

In [1]:
from pyspark.sql.functions import col, to_date, upper, trim

# The 5 delay-cause columns (minutes). We use this list in every later step.
CAUSE_COLS = ["carrier_delay", "weather_delay", "nas_delay",
              "security_delay", "late_aircraft_delay"]

flights_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(FLIGHTS_PATH)

print("Raw flights:", flights_raw.count())
flights_raw.printSchema()

ModuleNotFoundError: No module named 'pyspark'

1.2 Cleaning the flights file

In [ ]:
#  Keep only the columns we need
flights = flights_raw.select(
    "flight_date", "year", "month", "airline_code", "flight_number",
    "origin", "dest", "arr_delay", "cancelled", "diverted", *CAUSE_COLS
)

# Fix types and tidy the codes
flights = flights \
    .withColumn("flight_date",  to_date(col("flight_date"), "yyyy-MM-dd")) \
    .withColumn("airline_code", upper(trim(col("airline_code")))) \
    .withColumn("origin",       upper(trim(col("origin")))) \
    .withColumn("dest",         upper(trim(col("dest")))) \
    .withColumn("arr_delay",    col("arr_delay").cast("double")) \
    .withColumn("cancelled",    col("cancelled").cast("double")) \
    .withColumn("diverted",     col("diverted").cast("double"))

for cause in CAUSE_COLS:
    flights = flights.withColumn(cause, col(cause).cast("double"))

# A flight is identified by: date + airline + flight number + origin
FLIGHT_KEY = ["flight_date", "airline_code", "flight_number", "origin"]

# Remove unusable rows, missing values and duplicates
flights = flights \
    .where((col("cancelled") == 0) & (col("diverted") == 0)) \
    .dropna(subset=["flight_date", "airline_code", "dest", "arr_delay"]) \
    .na.fill(0, subset=CAUSE_COLS) \
    .dropDuplicates(FLIGHT_KEY) \
    .drop("cancelled", "diverted") \
    .withColumnRenamed("dest", "dest_airport")   # arrival airport = where arr_delay is measured

flights.cache()
print("Clean flights:", flights.count())
flights.show(5)

1.3 Read and clean the airlines and airports file

In [ ]:
# Airports: keep only the descriptive columns
airports = spark.read.format("csv").option("header", "true").load(AIRPORTS_PATH) \
    .select(upper(trim(col("iata_code"))).alias("iata_code"),
            "airport_name", "city", "state") \
    .dropna(subset=["iata_code"]) \
    .dropDuplicates(["iata_code"])

# Airlines: keep only the airline code and the full name
airlines = spark.read.format("csv").option("header", "true").load(AIRLINES_PATH) \
    .select(upper(trim(col("airline_code"))).alias("airline_code"), "airline") \
    .dropna(subset=["airline_code"]) \
    .dropDuplicates(["airline_code"])

print("Airports:", airports.count(), "| Airlines:", airlines.count())
airports.show(5)
airlines.show(5)

2. Keep only the late flights

In [ ]:
from pyspark.sql.functions import lit

# A flight counts as "late" when it arrives 15+ minutes late.
# Only these flights have cause values filled in.
late = flights.where(col("arr_delay") >= 15)

# Safety check: drop late flights where all 5 causes are 0 (no cause recorded)
total_cause = sum([col(cause) for cause in CAUSE_COLS], lit(0))
late = late.where(total_cause > 0)

print("Late flights:", late.count())
late.show(5)

3. Unpivot (5 cause columns to rows)

In [ ]:
from pyspark.sql.functions import regexp_replace

ID_COLS = FLIGHT_KEY + ["year", "month", "dest_airport"]

causes_long = late.unpivot(
    ids=ID_COLS,                      # columns that stay as they are
    values=CAUSE_COLS,                # the 5 columns that become rows
    variableColumnName="cause",       # new column: cause name
    valueColumnName="delay_minutes"   # new column: hold how many minutes delay from old column
)

# Remove rows where that cause contributed 0 minutes
causes_long = causes_long.where(col("delay_minutes") > 0)

# Make the cause names shorter: "weather_delay" -> "weather"
causes_long = causes_long.withColumn("cause", regexp_replace(col("cause"), "_delay$", ""))

print("Rows after unpivot:", causes_long.count())
causes_long.show(10)

In [ ]:
from pyspark.sql.functions import sum as spark_sum

# Check: do each flight's cause minutes add up to its arrival delay?
check = causes_long.groupBy(FLIGHT_KEY) \
    .agg(spark_sum("delay_minutes").alias("cause_total")) \
    .join(late.select(*FLIGHT_KEY, "arr_delay"), FLIGHT_KEY)

print("Flights checked:", check.count())
print("Flights where causes != arr_delay:",
      check.where(col("cause_total") != col("arr_delay")).count())

4. Join with airports and airlines

In [ ]:
# Join 1: add airport name, city, state  (dest_airport = iata_code)
joined = causes_long.join(
    airports,
    causes_long["dest_airport"] == airports["iata_code"],
    "left"
).drop("iata_code")      # same value as dest_airport, so we don't need it twice

# Join 2: add the airline's full name  (both tables have a column called airline_code)
joined = joined.join(airlines, ["airline_code"], "left")

# Airports/airlines missing from the lookup tables get "Unknown" instead of empty
joined = joined.na.fill("Unknown", subset=["airport_name", "city", "state", "airline"])

# Check: how many rows found no match?
print("Rows without airport match:", joined.where(col("airport_name") == "Unknown").count())
print("Rows without airline match:", joined.where(col("airline") == "Unknown").count())
joined.show(5)

Section 5: Share of each cause per airport and per airline, per month

In [ ]:
from pyspark.sql.functions import sum as spark_sum, count, round as spark_round
from pyspark.sql.window import Window

# 5a. Per AIRPORT per month 
airport_month = joined.groupBy(
    "dest_airport", "airport_name", "city", "state", "year", "month", "cause"
).agg(
    spark_sum("delay_minutes").alias("delay_minutes"),   # total minutes for this cause
    count("*").alias("num_late_flights")                 # how many late flights had this cause
)

w_airport = Window.partitionBy("dest_airport", "year", "month")   # one "group" = one airport in one month

airport_month = airport_month \
    .withColumn("total_delay_minutes", spark_sum("delay_minutes").over(w_airport)) \
    .withColumn("cause_share", spark_round(col("delay_minutes") / col("total_delay_minutes"), 4))

airport_month.orderBy("dest_airport", "year", "month", col("cause_share").desc()).show(10)

# 5b. Per AIRLINE per month 
airline_month = joined.groupBy(
    "airline_code", "airline", "year", "month", "cause"
).agg(
    spark_sum("delay_minutes").alias("delay_minutes"),
    count("*").alias("num_late_flights")
)

w_airline = Window.partitionBy("airline_code", "year", "month")   # one group = one airline in one month

airline_month = airline_month \
    .withColumn("total_delay_minutes", spark_sum("delay_minutes").over(w_airline)) \
    .withColumn("cause_share", spark_round(col("delay_minutes") / col("total_delay_minutes"), 4))

airline_month.orderBy("airline_code", "year", "month", col("cause_share").desc()).show(10)

6. Each airport's main cause, and its main cause per season

In [ ]:
from pyspark.sql.functions import when, row_number, first, concat, coalesce, max as spark_max

MIN_LATE_FLIGHTS = 100   # skip tiny airports: with very few late flights the shares aren't meaningful

# ---------- 6a. Add a season column ----------
joined_s = joined.withColumn(
    "season",
    when(col("month").isin(12, 1, 2), "Winter")
    .when(col("month").isin(3, 4, 5), "Spring")
    .when(col("month").isin(6, 7, 8), "Summer")
    .otherwise("Autumn")
)

# ---------- 6b. Top 2 causes per airport (whole period) ----------
airport_total = joined_s.groupBy("dest_airport", "airport_name", "city", "state", "cause") \
    .agg(spark_sum("delay_minutes").alias("delay_minutes"))

w_ap = Window.partitionBy("dest_airport")                        # group = one airport
airport_total = airport_total.withColumn(
    "cause_share",
    spark_round(col("delay_minutes") / spark_sum("delay_minutes").over(w_ap), 4)
)

w_rank = Window.partitionBy("dest_airport").orderBy(col("delay_minutes").desc(), col("cause"))

top2 = airport_total \
    .withColumn("rank", row_number().over(w_rank)) \
    .where(col("rank") <= 2)

main_overall = top2.groupBy("dest_airport", "airport_name", "city", "state").agg(
    spark_max(when(col("rank") == 1, col("cause"))).alias("main_cause"),
    spark_max(when(col("rank") == 1, col("cause_share"))).alias("main_cause_share"),
    spark_max(when(col("rank") == 2, col("cause"))).alias("second_cause"),
    spark_max(when(col("rank") == 2, col("cause_share"))).alias("second_cause_share")
)

# ---------- 6c. Main cause per airport per season ----------
airport_season = joined_s.groupBy("dest_airport", "season", "cause") \
    .agg(spark_sum("delay_minutes").alias("delay_minutes"))

w_aps = Window.partitionBy("dest_airport", "season")             # group = one airport in one season
airport_season = airport_season.withColumn(
    "cause_share",
    spark_round(col("delay_minutes") / spark_sum("delay_minutes").over(w_aps), 4)
)

w_rank_s = Window.partitionBy("dest_airport", "season").orderBy(col("delay_minutes").desc(), col("cause"))
main_season = airport_season \
    .withColumn("rank", row_number().over(w_rank_s)) \
    .where(col("rank") == 1)

# ---------- 6d. Pivot: seasons become columns ----------
season_wide = main_season.groupBy("dest_airport") \
    .pivot("season", ["Winter", "Spring", "Summer", "Autumn"]) \
    .agg(first("cause").alias("cause"), first("cause_share").alias("share"))

# ---------- 6e. Number of late flights per airport ----------
late_counts = late.groupBy("dest_airport").agg(count("*").alias("num_late_flights"))

# ---------- 6f. Peak season of the main cause ----------
peak = airport_season.join(
    main_overall.select("dest_airport", col("main_cause").alias("cause")),
    ["dest_airport", "cause"],
    "inner"
)

w_peak = Window.partitionBy("dest_airport").orderBy(col("cause_share").desc(), col("season"))

peak = peak \
    .withColumn("r", row_number().over(w_peak)) \
    .where(col("r") == 1) \
    .select("dest_airport",
            col("season").alias("peak_season"),
            col("cause_share").alias("peak_season_share"))

# ---------- 6g. Combine everything into one summary row per airport ----------
def pct(c):
    """0.3512 -> '35'"""
    return (col(c) * 100).cast("int").cast("string")

airport_summary = main_overall \
    .join(late_counts, ["dest_airport"], "left") \
    .join(season_wide, ["dest_airport"], "left") \
    .join(peak, ["dest_airport"], "left") \
    .where(col("num_late_flights") >= MIN_LATE_FLIGHTS) \
    .withColumn(
        "summary",
        concat(
            col("city"), lit(": mostly "), col("main_cause"),
            lit(" ("), pct("main_cause_share"), lit("%) and "),
            coalesce(col("second_cause"), lit("n/a")),
            lit(" ("), coalesce(pct("second_cause_share"), lit("-")), lit("%), especially in "),
            coalesce(col("peak_season"), lit("n/a")),
            lit(" ("), coalesce(pct("peak_season_share"), lit("-")), lit("%)")
        )
    )

print("Airports in summary:", airport_summary.count())
airport_summary.orderBy(col("num_late_flights").desc()) \
    .select("dest_airport", "num_late_flights", "main_cause", "main_cause_share",
            "second_cause", "second_cause_share", "peak_season", "summary") \
    .show(15, truncate=False)

7. Write the results to bigquery

7.1. Choose the final columns

In [ ]:
# Put the columns in a clear order for each output table
out_airport = airport_month.select(
    "dest_airport", "airport_name", "city", "state", "year", "month",
    "cause", "delay_minutes", "num_late_flights", "total_delay_minutes", "cause_share"
)

out_airline = airline_month.select(
    "airline_code", "airline", "year", "month",
    "cause", "delay_minutes", "num_late_flights", "total_delay_minutes", "cause_share"
)

out_summary = airport_summary.select(
    "dest_airport", "airport_name", "city", "state", "num_late_flights",
    "main_cause", "main_cause_share", "second_cause", "second_cause_share",
    "peak_season", "peak_season_share",
    "Winter_cause", "Winter_share", "Spring_cause", "Spring_share",
    "Summer_cause", "Summer_share", "Autumn_cause", "Autumn_share",
    "summary"
)



7.2 Write them out (Bigquery on the VM)

In [ ]:
def write_output(df, table_name):
    """Write a DataFrame to BigQuery."""
    df.write.format("bigquery") \
        .option("table", table_name) \
        .mode("overwrite") \
        .save()
    print(f"Written to BigQuery: {table_name}")

write_output(out_airport, TABLE_AIRPORT)
write_output(out_airline, TABLE_AIRLINE)
write_output(out_summary, TABLE_SUMMARY)

7.3. stopping

In [ ]:
flights.unpersist()   # free the cached data from Section 1
spark.stop()          # stop the Spark session